# Day 51: Ensemble Methods - Bagging & Boosting

## Bagging
Ensemble methods combine many individually weak or noisy models into one stronger predictor, and bagging (short for Bootstrap Aggregating) is the "train many models independently, then combine" family. Each model is trained on a different random bootstrap sample of the training data (sampled with replacement, so each sample is roughly the same size as the original but contains duplicates and omits some rows), which makes each individual model slightly different and prone to different mistakes. Averaging their predictions (for regression) or taking a majority vote (for classification) smooths out those individual quirks, which specifically reduces variance - it doesn't help much with a model that's fundamentally too simple (bias), but it substantially helps a model that's prone to overfitting on any single sample of data, like an unrestricted decision tree. `BaggingClassifier` here wraps 50 separate decision trees, each trained on its own bootstrap sample, and the final prediction is their majority vote - this is conceptually what a Random Forest does, just without the extra random-feature-subset trick Random Forest adds.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

bagging = BaggingClassifier(DecisionTreeClassifier(), n_estimators=50, random_state=42)  # 50 trees, each on a bootstrap sample, combined by majority vote
bagging.fit(X_train, y_train)
print('Bagging accuracy:', bagging.score(X_test, y_test))

## AdaBoost
Boosting takes the opposite approach from bagging: instead of training many models independently and in parallel, it trains models sequentially, where each new model is deliberately built to fix the mistakes of the ensemble so far. AdaBoost (Adaptive Boosting) does this by keeping a weight for every training sample; after each round, it increases the weight of samples the current ensemble got wrong (so the next weak learner - typically a shallow "decision stump" - is forced to pay more attention to those hard cases) and decreases the weight of samples already handled correctly. The final prediction is a weighted vote of all the sequential weak learners, where more accurate learners get more say. Because each step actively targets the previous errors, boosting tends to reduce bias effectively (turning a collection of weak, simple models into a strong one), though it can be more sensitive to noisy data/outliers than bagging since it keeps re-emphasizing hard-to-classify points. `n_estimators=50` here means 50 rounds of sequential weak learners are combined.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=50, random_state=42)  # 50 sequential weak learners, reweighting hard samples each round
ada.fit(X_train, y_train)
print('AdaBoost accuracy:', ada.score(X_test, y_test))

## Gradient Boosting
Gradient Boosting is another sequential boosting method, but instead of reweighting misclassified samples like AdaBoost, each new model is trained to directly predict the residual errors (the gap between the current ensemble's predictions and the true values) left by the models before it - effectively performing gradient descent in "prediction space" to minimize the overall loss function. Each new tree's predictions get added to the ensemble scaled by the `learning_rate`, so `learning_rate=0.1` means each new tree only nudges the overall prediction a little, requiring more trees (`n_estimators=100`) but generally producing a smoother, less overfit-prone result than large steps would. This approach (and its modern, faster implementations like XGBoost and LightGBM, covered on Day 52) is widely regarded as one of the most accurate methods for structured/tabular data. As with AdaBoost, accuracy here should be compared against bagging to see which approach - reducing variance vs. reducing bias - helps more on this particular dataset.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)  # 100 trees, each correcting the residual error of the ensemble so far, added in small (0.1) steps
gb.fit(X_train, y_train)
print('Gradient Boosting accuracy:', gb.score(X_test, y_test))

## Bagging vs Boosting
It's worth summarizing the core distinction clearly, since it's easy to blur the two together. Bagging trains many models in parallel and independently (each on its own random bootstrap sample) and combines them by averaging/voting - this mainly attacks variance, making an already-decent-but-unstable model (like a deep decision tree) more stable and reliable, and it parallelizes trivially since the models don't depend on each other. Boosting trains models sequentially, where each one is built specifically to address the shortcomings of the ensemble so far - this mainly attacks bias, turning a collection of weak/simple learners into a strong one, but it's inherently sequential (harder to parallelize) and more sensitive to overfitting/noisy data if not tuned carefully (via learning rate, number of estimators, tree depth, etc). Knowing which problem you have (high bias vs high variance, from Day 50) helps you pick the right family of ensemble method.

In [ ]:
print('Bagging: parallel, reduces variance, e.g. Random Forest')
print('Boosting: sequential, reduces bias, e.g. AdaBoost, Gradient Boosting, XGBoost')